# Chapter 12: Cross-Validation and Model Evaluation
## 📌 Summary
Evaluasi model yang benar adalah kunci ML yang reliable. Chapter ini mencakup **cross-validation, hyperparameter tuning (GridSearch, RandomSearch), berbagai metrics**, dan **learning curves**.

## 🧠 Theoretical Deep-Dive

### 12.1 Cross-Validation
**Problem dengan simple train-test split**: hasil bergantung pada random seed; tidak reliable.

**K-Fold CV**: bagi data menjadi k folds; latih k kali, setiap kali 1 fold sebagai test.
- Score CV = rata-rata k scores → lebih reliable estimate of generalization
- `StratifiedKFold`: preserves class distribution per fold

### 12.2 GridSearchCV vs RandomizedSearchCV
- **GridSearchCV**: coba semua kombinasi → exhaustive, expensive untuk banyak hyperparameter
- **RandomizedSearchCV**: sample n kombinasi random → lebih efisien, sering cukup baik

### 12.3 Metrics untuk Classification
- **Accuracy**: (TP+TN)/Total → misleading untuk imbalanced
- **Precision**: TP/(TP+FP) → minimasi false positives
- **Recall**: TP/(TP+FN) → minimasi false negatives
- **F1**: 2×(P×R)/(P+R) → harmonic mean, balance P & R
- **ROC-AUC**: area under ROC curve, threshold-independent

### 12.4 Metrics untuk Regression
- **MAE**: mean absolute error → robust terhadap outlier
- **MSE/RMSE**: mean squared error → penalizes large errors lebih
- **R²**: explained variance ratio, [0,1]

## 💻 Code Reproduction

In [ ]:
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
import numpy as np

X, y = load_breast_cancer(return_X_y=True)
clf = RandomForestClassifier(n_estimators=50, random_state=42)

# Basic cross_val_score
cv_scores = cross_val_score(clf, X, y, cv=5, scoring="accuracy")
print("5-Fold CV scores:", cv_scores.round(4))
print(f"Mean: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

# StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_s = cross_val_score(clf, X, y, cv=skf, scoring="accuracy")
print("\nStratified 5-Fold CV:", cv_scores_s.round(4))
print(f"Mean: {cv_scores_s.mean():.4f} ± {cv_scores_s.std():.4f}")

# Multiple metrics
result = cross_validate(clf, X, y, cv=5, scoring=["accuracy", "f1", "roc_auc"])
for metric, values in result.items():
    if metric.startswith("test_"):
        print(f"{metric}: {values.mean():.4f} ± {values.std():.4f}")

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import pandas as pd

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5, 
                           scoring="roc_auc", n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_.round(4))
print("Test ROC-AUC:", grid_search.score(X_test, y_test).round(4))

# Top 5 results
results = pd.DataFrame(grid_search.cv_results_)
print("\nTop 5 CV results:")
print(results.nlargest(5, "mean_test_score")[["params", "mean_test_score", "std_test_score"]].to_string())

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from scipy.stats import uniform, randint

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

param_dist = {
    "n_estimators": randint(50, 300),
    "learning_rate": uniform(0.01, 0.3),
    "max_depth": randint(2, 7),
    "subsample": uniform(0.6, 0.4)
}

random_search = RandomizedSearchCV(GradientBoostingClassifier(random_state=42), param_dist, 
                                    n_iter=30, cv=5, scoring="roc_auc", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV ROC-AUC:", random_search.best_score_.round(4))
print("Test ROC-AUC:", random_search.score(X_test, y_test).round(4))

In [ ]:
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                              precision_recall_curve, average_precision_score,
                              mean_absolute_error, mean_squared_error, r2_score)
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.datasets import load_breast_cancer, make_regression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# Classification metrics
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba).round(4))
print("Avg Precision:", average_precision_score(y_test, y_proba).round(4))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Regression metrics
X_r, y_r = make_regression(n_samples=200, n_features=5, noise=20, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train_r, y_train_r)
y_pred_r = reg.predict(X_test_r)
print("\nRegression Metrics:")
print("MAE:", mean_absolute_error(y_test_r, y_pred_r).round(2))
print("RMSE:", np.sqrt(mean_squared_error(y_test_r, y_pred_r)).round(2))
print("R²:", r2_score(y_test_r, y_pred_r).round(4))

In [ ]:
from sklearn.model_selection import learning_curve, validation_curve
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
import numpy as np
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)

# Learning Curve
train_sizes, train_scores, val_scores = learning_curve(
    RandomForestClassifier(n_estimators=50, random_state=42), X, y,
    train_sizes=np.linspace(0.1, 1.0, 10), cv=5, scoring="accuracy", n_jobs=-1
)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(train_sizes, train_scores.mean(1), label="Train", marker="o")
plt.fill_between(train_sizes, train_scores.mean(1)-train_scores.std(1), train_scores.mean(1)+train_scores.std(1), alpha=0.2)
plt.plot(train_sizes, val_scores.mean(1), label="Val", marker="s")
plt.fill_between(train_sizes, val_scores.mean(1)-val_scores.std(1), val_scores.mean(1)+val_scores.std(1), alpha=0.2)
plt.xlabel("Training size"); plt.ylabel("Accuracy"); plt.title("Learning Curve"); plt.legend()

# Validation Curve
param_range = [1, 5, 10, 20, 50, 100, 200]
train_s, val_s = validation_curve(
    RandomForestClassifier(random_state=42), X, y,
    param_name="n_estimators", param_range=param_range, cv=5, scoring="accuracy"
)

plt.subplot(1, 2, 2)
plt.plot(param_range, train_s.mean(1), label="Train", marker="o")
plt.plot(param_range, val_s.mean(1), label="Val", marker="s")
plt.xlabel("n_estimators"); plt.ylabel("Accuracy"); plt.title("Validation Curve"); plt.legend()
plt.tight_layout(); plt.show()

## ✅ Chapter Summary
- **StratifiedKFold** → default untuk classification, preserves class distribution
- **GridSearchCV** → exhaustive, gunakan untuk ≤4 hyperparameters
- **RandomizedSearchCV** → efisien untuk banyak hyperparameters & continuous distributions
- **Accuracy misleading** untuk imbalanced data → gunakan F1, ROC-AUC, atau PR-AUC
- **Learning curve** → diagnose underfitting (both low) vs overfitting (gap train-val)
- **Nested CV** untuk unbiased estimate: outer CV untuk evaluate, inner CV untuk tuning